# 🤖 Giai đoạn 3: Huấn luyện & So sánh các mô hình Machine Learning
Thử nghiệm Naive Bayes, Logistic Regression, SVM, Random Forest & Stacking Classifier


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import load_feature_split
from src.models import SentimentModelTrainer

artifact = load_feature_split(PROJECT_ROOT / 'models' / 'train_test_features.joblib')
if artifact['metadata'].get('feature_mode') != 'text_only':
    raise ValueError('Notebook TV3 yêu cầu artifact text-only.')
X_train, X_test = artifact['X_train'], artifact['X_test']
y_train, y_test = artifact['y_train'], artifact['y_test']
print(f'Train: {X_train.shape}; final test: {X_test.shape}')
print('Final test chỉ được dùng sau khi đã chọn xong mô hình bằng CV trên train.')


## 3.1 Chọn mô hình bằng Cross-Validation trên tập train


In [ ]:
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score

trainer = SentimentModelTrainer()
candidates = {**trainer.models, 'Stacking': trainer.get_stacking_model()}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
cv_rows = []
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    cv_rows.append({'Model': name, 'CV Macro F1 Mean': scores.mean(), 'CV Macro F1 Std': scores.std()})
cv_results = pd.DataFrame(cv_rows).sort_values('CV Macro F1 Mean', ascending=False)
best_model_name = cv_results.iloc[0]['Model']
display(cv_results.round(4))
print(f'Mô hình được chọn trên train CV: {best_model_name}')


## 3.2 Fit mô hình đã chọn và đánh giá final test đúng một lần


In [ ]:
from sklearn.base import clone
from sklearn.metrics import classification_report, f1_score

best_model = clone(candidates[best_model_name])
best_model.fit(X_train, y_train)
final_prediction = best_model.predict(X_test)
print(f'Final-test Macro F1: {f1_score(y_test, final_prediction, average="macro"):.4f}')
print(classification_report(y_test, final_prediction, digits=4))
trainer.trained_models[best_model_name] = best_model
trainer.save_model(best_model_name, PROJECT_ROOT / 'models' / 'best_text_sentiment_model.joblib')
